# 11 -- VIX Futures Term Structure & CBOE SKEW Index Data Collection

## Purpose
Collects two orthogonal market-level signals not captured elsewhere in the pipeline:
1. **VIX Futures Term Structure** -- the spread between second-month and front-month VIX futures. Positive = contango (normal), negative = backwardation (panic). Distinct from VIX spot (already collected in notebook 09) because it captures how the market prices future volatility relative to current volatility.
2. **CBOE SKEW Index** -- measures perceived tail risk of the S&P 500 by pricing the difference between OTM puts and OTM calls on SPX. SKEW = 100 means log-normal (no tail risk); values above 100 indicate the market is paying a premium for crash protection.

## Sources

### VIX Futures
WRDS LSEG Datastream Futures (`tr_ds_fut.dsfutcalcserval`) via the `wrds` Python library, authenticated with username `henrylavender`. Data pulled from 2004-01-01 to 2024-12-31.

**Series code selection:** A diagnostic script was run to evaluate date coverage across all available VIX-related `calcseriescode` values in the table. The initial attempt used codes 13396 (labelled "CFE-VIX INDEX CONTINUOUS") and 22013 (labelled "CFE-VIX INDEX CONT. 2ND FUT"), but these only contained data from May 2024 onwards -- they are a recently created series. The corrected version uses:
- `calcseriescode = 17679` -- CFE-VIX INDEX TRc1 (front-month continuous), Thomson Reuters continuous contract series stitched from individual expiring contracts, full history from 2004-03-26
- `calcseriescode = 17680` -- CFE-VIX INDEX TRc2 (second-month continuous), same construction, full history from 2004-03-26

The initial version also included a yfinance fallback (`VX=F` ticker) that would trigger if WRDS coverage was insufficient; this fallback is no longer needed with the correct series codes.

### CBOE SKEW
Locally downloaded CSV (`SKEW_History.csv`) from cboe.com. Columns: `DATE`, `SKEW`.

## Collection Method

### VIX Futures
Front-month and second-month settlement prices, volume, and open interest are pulled directly from `tr_ds_fut.dsfutcalcserval` (clean 8-column schema, one row per calcseriescode per date). The date column is `date_` and the price column is `settlement`. Both series are validated to have 4,000+ rows (expected ~5,000+ for 2004--2024). Rows with NaN or zero settlement are dropped. Front and second-month series are outer-joined on date.

### CBOE SKEW
The CSV is read, `DATE` is parsed to datetime, `SKEW` is cast to numeric, rows with invalid dates or values are dropped, and the result is filtered to 2004-01-01 through 2024-12-31.

## Variables Collected

### VIX Futures (Raw)
- `vix_fut_front` -- front-month VIX futures settlement price
- `vix_fut_second` -- second-month VIX futures settlement price
- `vix_fut_volume` -- front-month daily volume (contracts)
- `vix_fut_oi` -- front-month daily open interest (contracts)

### VIX Futures (Derived)
- `vix_term_spread` -- VF2 - VF1 (contango > 0, backwardation < 0)
- `vix_term_ratio` -- VF2 / VF1 (normalised term structure; >1 = contango, <1 = backwardation)
- `vix_fut_ret_1d` -- daily percentage return on front-month VIX futures
- `vix_term_spread_5d_chg` -- 5-day first difference of the term structure spread

### CBOE SKEW (Raw)
- `skew` -- CBOE SKEW Index level (typical range ~110--160)

### CBOE SKEW (Derived)
- `skew_excess` -- SKEW minus 100 (tail risk premium above log-normal baseline)
- `skew_pctile_252d` -- rolling 1-year (252-day, min 126) percentile of SKEW, computed as the fraction of the trailing window that the current value exceeds
- `skew_chg_5d` -- 5-day first difference of SKEW
- `skew_ma20` -- 20-day moving average of SKEW (min 10 observations)
- `skew_vs_ma20` -- SKEW minus its 20-day moving average

## Pipeline Notes
- All data is daily and market-level (no stock dimension). Merge onto trading calendar by date.
- VIX futures basis (`vix_fut_front - VIX_spot`) should be computed in the downstream merge pipeline where `macro_daily.parquet` (containing the VIX spot close) is available.
- No look-ahead concerns: all values are known at market close of date t. SKEW derived factors use backward-looking windows only.
- VIX futures launched March 26, 2004. Data before that date will be NaN.

## Outputs
Saved to `Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/`:
- `vix_futures_daily.parquet` -- VIX futures panel (raw + derived)
- `cboe_skew_daily.parquet` -- CBOE SKEW panel (raw + derived)
- `vix_skew_combined.parquet` -- outer join of VIX futures and SKEW on date (all factors in one file)

In [4]:
# %% [markdown]
# # Stage 11: VIX Futures Term Structure (WRDS) & CBOE SKEW Index
#
# Two orthogonal signals not captured anywhere else in the pipeline:
#
# 1. **VIX Futures Term Structure** — the spread between 2nd-month and front-month
#    VIX futures. Positive = contango (normal), negative = backwardation (panic).
#    This is distinct from VIX spot (which you already have) because it captures
#    how the market prices FUTURE volatility relative to CURRENT volatility.
#
# 2. **CBOE SKEW Index** — measures the perceived tail risk of the S&P 500 by
#    pricing the difference between OTM puts and OTM calls on SPX. High SKEW
#    = market is paying a premium for crash protection.
#
# Source tables:
#   WRDS: tr_ds_fut.dsfutcalcserval (primary — clean 8-column schema)
#     - calcseriescode 13396 = CFE-VIX INDEX CONTINUOUS (front-month)
#     - calcseriescode 22013 = CFE-VIX INDEX CONT. 2ND FUT (second-month)
#   Local CSV: CBOE SKEW_History.csv (downloaded from cboe.com)
#
# Output:
#   Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/vix_futures_daily.parquet
#   Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/cboe_skew_daily.parquet
#   Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/vix_skew_combined.parquet

# %% [markdown]
# ## Setup

# %%
import wrds
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

conn = wrds.Connection(wrds_username='henrylavender')

START = '2004-01-01'
END   = '2024-12-31'

# %% [markdown]
# ---
# # PART 1: VIX FUTURES TERM STRUCTURE
# ---

# %% [markdown]
# ## Step 1: Schema verification
#
# Confirm the table structure before pulling. We use `dsfutcalcserval` as the
# primary table — it has a clean 8-column schema with no metadata rows.
# The date column is `date_` (NOT `marketdate`).
#
# The alternative `wrds_fut_series` table has 25 columns including metadata
# rows that inflate the row count by ~5x and only contains recent data for
# some series. We avoid it.

# %%
print("=" * 80)
print("SCHEMA VERIFICATION: tr_ds_fut.dsfutcalcserval")
print("=" * 80)

schema = conn.describe_table('tr_ds_fut', 'dsfutcalcserval')
print(f"\ntr_ds_fut.dsfutcalcserval — {len(schema)} columns:")
for _, row in schema.iterrows():
    print(f"  {row['name']:<20s} {str(row['type']):<20s}")

# Quick sample to verify data looks right
sample = conn.raw_sql("""
    SELECT *
    FROM tr_ds_fut.dsfutcalcserval
    WHERE calcseriescode = 13396
    ORDER BY date_ DESC
    LIMIT 5
""")
print(f"\nSample rows (calcseriescode=13396, front-month VIX, most recent):")
print(sample.to_string(index=False))

# Check date range available
date_range = conn.raw_sql("""
    SELECT MIN(date_) AS min_date, MAX(date_) AS max_date, COUNT(*) AS n_rows
    FROM tr_ds_fut.dsfutcalcserval
    WHERE calcseriescode = 13396
""")
print(f"\nFull date range for front-month VIX (13396):")
print(date_range.to_string(index=False))

date_range_2 = conn.raw_sql("""
    SELECT MIN(date_) AS min_date, MAX(date_) AS max_date, COUNT(*) AS n_rows
    FROM tr_ds_fut.dsfutcalcserval
    WHERE calcseriescode = 22013
""")
print(f"\nFull date range for second-month VIX (22013):")
print(date_range_2.to_string(index=False))

# %% [markdown]
# ## Step 2: Pull VIX futures data
#
# Pull front-month (13396) and second-month (22013) continuous VIX futures
# from `dsfutcalcserval`. This table has one clean row per (calcseriescode, date_).

# %%
print("=" * 80)
print("PULLING VIX FUTURES FROM tr_ds_fut.dsfutcalcserval")
print("=" * 80)

# ── Front-month continuous VIX futures ───────────────────────────────────────
print("\nPulling front-month (calcseriescode=13396)...")
vix_front_raw = conn.raw_sql(f"""
    SELECT date_, open_, high, low, volume, settlement, openinterest
    FROM tr_ds_fut.dsfutcalcserval
    WHERE calcseriescode = 13396
      AND date_ BETWEEN '{START}' AND '{END}'
    ORDER BY date_
""")
print(f"  Rows: {len(vix_front_raw):,}")
if len(vix_front_raw) > 0:
    print(f"  Date range: {vix_front_raw['date_'].min()} → {vix_front_raw['date_'].max()}")
    print(f"  Settlement range: {vix_front_raw['settlement'].min():.2f} – "
          f"{vix_front_raw['settlement'].max():.2f}")

# ── Second-month continuous VIX futures ──────────────────────────────────────
print("\nPulling second-month (calcseriescode=22013)...")
vix_second_raw = conn.raw_sql(f"""
    SELECT date_, open_, high, low, volume, settlement, openinterest
    FROM tr_ds_fut.dsfutcalcserval
    WHERE calcseriescode = 22013
      AND date_ BETWEEN '{START}' AND '{END}'
    ORDER BY date_
""")
print(f"  Rows: {len(vix_second_raw):,}")
if len(vix_second_raw) > 0:
    print(f"  Date range: {vix_second_raw['date_'].min()} → {vix_second_raw['date_'].max()}")
    print(f"  Settlement range: {vix_second_raw['settlement'].min():.2f} – "
          f"{vix_second_raw['settlement'].max():.2f}")

# ── Validate we got enough data ──────────────────────────────────────────────
MIN_EXPECTED_ROWS = 2000  # ~8 years minimum

if len(vix_front_raw) < MIN_EXPECTED_ROWS or len(vix_second_raw) < MIN_EXPECTED_ROWS:
    print(f"\n⚠ WARNING: Expected at least {MIN_EXPECTED_ROWS} rows per contract "
          f"(got {len(vix_front_raw)} front, {len(vix_second_raw)} second).")
    print("  The LSEG Datastream subscription on WRDS may not have full VIX futures history.")
    print("  Consider the yfinance fallback for front-month (see Part 1b below).")
    COVERAGE_OK = False
else:
    print(f"\n✓ Coverage looks good: {len(vix_front_raw):,} front-month rows, "
          f"{len(vix_second_raw):,} second-month rows.")
    COVERAGE_OK = True

# %% [markdown]
# ## Step 2b: yfinance fallback (only runs if WRDS coverage is insufficient)
#
# If the WRDS LSEG Datastream subscription doesn't have full VIX futures history,
# we fall back to Yahoo Finance for the front-month continuous series.
# Yahoo Finance tickers:
#   - ^VIX  = VIX spot index (not what we want — we already have this)
#   - VX=F  = Front-month VIX futures continuous
#
# Note: yfinance does NOT have a clean second-month VIX futures series.
# If we need it, we'd have to use the CBOE historical data CSVs instead.

# %%
if not COVERAGE_OK:
    print("=" * 80)
    print("YFINANCE FALLBACK: Pulling VIX futures from Yahoo Finance")
    print("=" * 80)

    try:
        import yfinance as yf

        # Front-month VIX futures
        print("\nDownloading VX=F (front-month VIX futures)...")
        vx = yf.download("VX=F", start=START, end=END, progress=False)

        if len(vx) > MIN_EXPECTED_ROWS:
            print(f"  ✓ Got {len(vx):,} rows from yfinance")
            print(f"  Date range: {vx.index.min().date()} → {vx.index.max().date()}")

            # Reshape to match our schema
            vix_front_raw = pd.DataFrame({
                'date_': vx.index,
                'open_': vx['Open'].values,
                'high': vx['High'].values,
                'low': vx['Low'].values,
                'volume': vx['Volume'].values,
                'settlement': vx['Close'].values,  # yfinance uses Close, not Settlement
                'openinterest': np.nan,  # yfinance doesn't provide OI
            })
            vix_front_raw['date_'] = pd.to_datetime(vix_front_raw['date_'])

            print(f"  Settlement range: {vix_front_raw['settlement'].min():.2f} – "
                  f"{vix_front_raw['settlement'].max():.2f}")

            # For second-month, we won't have it from yfinance
            # Check if WRDS at least had some second-month data
            if len(vix_second_raw) < 100:
                print("\n  ⚠ No second-month VIX futures available from either source.")
                print("    Term structure factors will be limited to the WRDS date range.")
            else:
                print(f"\n  Using WRDS second-month data ({len(vix_second_raw)} rows) "
                      f"alongside yfinance front-month.")

            COVERAGE_OK = True
        else:
            print(f"  ⚠ yfinance also returned insufficient data ({len(vx)} rows)")

    except ImportError:
        print("  yfinance not installed. Run: pip install yfinance")
        print("  Proceeding with whatever WRDS data we have.")
    except Exception as e:
        print(f"  yfinance failed: {e}")
        print("  Proceeding with whatever WRDS data we have.")

# %% [markdown]
# ## Step 3: Build the VIX futures panel

# %%
print("=" * 80)
print("BUILDING VIX FUTURES PANEL")
print("=" * 80)

# ── Clean front-month ────────────────────────────────────────────────────────
vf1 = pd.DataFrame()
vf1['date'] = pd.to_datetime(vix_front_raw['date_'])
vf1['vix_fut_front'] = pd.to_numeric(vix_front_raw['settlement'], errors='coerce')

# Also grab volume and OI from the main table (dsfutcalcserval has them)
vf1['vix_fut_volume'] = pd.to_numeric(vix_front_raw['volume'], errors='coerce')
vf1['vix_fut_oi'] = pd.to_numeric(vix_front_raw['openinterest'], errors='coerce')

# Drop rows where settlement is NaN or zero (bad data)
vf1 = vf1[vf1['vix_fut_front'].notna() & (vf1['vix_fut_front'] > 0)]
vf1 = vf1.drop_duplicates(subset='date', keep='first')

print(f"Front-month cleaned: {len(vf1):,} rows")
print(f"  Date range: {vf1['date'].min().date()} → {vf1['date'].max().date()}")

# ── Clean second-month ───────────────────────────────────────────────────────
vf2 = pd.DataFrame()
vf2['date'] = pd.to_datetime(vix_second_raw['date_'])
vf2['vix_fut_second'] = pd.to_numeric(vix_second_raw['settlement'], errors='coerce')

vf2 = vf2[vf2['vix_fut_second'].notna() & (vf2['vix_fut_second'] > 0)]
vf2 = vf2.drop_duplicates(subset='date', keep='first')

print(f"Second-month cleaned: {len(vf2):,} rows")
if len(vf2) > 0:
    print(f"  Date range: {vf2['date'].min().date()} → {vf2['date'].max().date()}")

# ── Merge on date ────────────────────────────────────────────────────────────
vix_futures = vf1.merge(vf2, on='date', how='outer').sort_values('date').reset_index(drop=True)

print(f"\nMerged panel: {len(vix_futures):,} rows")
print(f"  Date range: {vix_futures['date'].min().date()} → {vix_futures['date'].max().date()}")

# %% [markdown]
# ## Step 4: Compute derived factors

# %%
# ── VIX futures term structure spread ────────────────────────────────────────
# Positive = contango (normal market, fear is priced as transient)
# Negative = backwardation (panic, near-term fear exceeds longer-term)
vix_futures['vix_term_spread'] = (
    vix_futures['vix_fut_second'] - vix_futures['vix_fut_front']
)

# ── VIX futures term structure ratio ─────────────────────────────────────────
# Normalised measure: VF2/VF1. >1 = contango, <1 = backwardation.
# More stable than the spread for cross-regime comparison.
vix_futures['vix_term_ratio'] = (
    vix_futures['vix_fut_second'] /
    vix_futures['vix_fut_front'].replace(0, np.nan)
)

# ── Daily return on front-month VIX futures ──────────────────────────────────
# Useful as a "vol momentum" signal
vix_futures['vix_fut_ret_1d'] = vix_futures['vix_fut_front'].pct_change()

# ── 5-day rolling change in term structure ───────────────────────────────────
vix_futures['vix_term_spread_5d_chg'] = vix_futures['vix_term_spread'].diff(5)

# ── Filter to sample period ──────────────────────────────────────────────────
vix_futures = vix_futures[
    (vix_futures['date'] >= START) & (vix_futures['date'] <= END)
].reset_index(drop=True)

print(f"VIX futures panel (final): {vix_futures.shape}")
print(f"Date range: {vix_futures['date'].min().date()} → {vix_futures['date'].max().date()}")
print(f"Columns: {vix_futures.columns.tolist()}")

# %% [markdown]
# ## Step 5: VIX futures sanity checks

# %%
print("=" * 80)
print("VIX FUTURES SANITY CHECKS")
print("=" * 80)

for col in ['vix_fut_front', 'vix_fut_second']:
    vals = vix_futures[col].dropna()
    if len(vals) > 0:
        print(f"\n{col} ({len(vals):,} valid obs):")
        print(f"  Mean:   {vals.mean():.2f}")
        print(f"  Median: {vals.median():.2f}")
        print(f"  Min:    {vals.min():.2f}")
        print(f"  Max:    {vals.max():.2f}")
    else:
        print(f"\n{col}: NO VALID DATA")

print(f"\nvix_term_spread (should be ~0-3 on average, negative in panics):")
ts = vix_futures['vix_term_spread'].dropna()
if len(ts) > 0:
    print(f"  Mean:   {ts.mean():.3f}")
    print(f"  Median: {ts.median():.3f}")
    print(f"  Min:    {ts.min():.3f}  (should be deeply negative in crisis)")
    print(f"  Max:    {ts.max():.3f}")

print(f"\nvix_term_ratio (should be ~1.0-1.1 on average, <1 in panics):")
tr = vix_futures['vix_term_ratio'].dropna()
if len(tr) > 0:
    print(f"  Mean:   {tr.mean():.4f}")
    print(f"  Median: {tr.median():.4f}")
    print(f"  Min:    {tr.min():.4f}")
    print(f"  Max:    {tr.max():.4f}")

print(f"\nNull counts:")
for c in vix_futures.columns:
    if c == 'date':
        continue
    n = vix_futures[c].isna().sum()
    pct = n / len(vix_futures) * 100
    print(f"  {c:<30s} {n:>5d} ({pct:.1f}%)")

# ── Crisis spot-checks ──────────────────────────────────────────────────────
for label, start_d, end_d in [
    ('GFC Sep-Oct 2008',       '2008-09-01', '2008-10-31'),
    ('Volmageddon Feb 2018',   '2018-02-01', '2018-02-15'),
    ('COVID Mar 2020',         '2020-03-01', '2020-03-31'),
    ('Yen carry unwind Aug 24','2024-08-01', '2024-08-15'),
]:
    mask = (vix_futures['date'] >= start_d) & (vix_futures['date'] <= end_d)
    subset = vix_futures[mask]
    if len(subset) > 0 and subset['vix_term_spread'].notna().any():
        print(f"\n{label}:")
        print(f"  vix_fut_front: {subset['vix_fut_front'].min():.1f} – "
              f"{subset['vix_fut_front'].max():.1f}")
        print(f"  vix_term_spread: {subset['vix_term_spread'].min():.2f} – "
              f"{subset['vix_term_spread'].max():.2f}")
        print(f"  (Expect backwardation = negative spread during crises)")
    else:
        print(f"\n{label}: no data in this range")

# %% [markdown]
# ## Step 6: Save VIX futures

# %%
vix_futures.to_parquet(
    OUTPUT_DIR / 'vix_futures_daily.parquet', index=False, engine='pyarrow'
)
print(f"Saved {OUTPUT_DIR / 'vix_futures_daily.parquet'}: {vix_futures.shape}")


# %% [markdown]
# ---
# # PART 2: CBOE SKEW INDEX
# ---

# %% [markdown]
# ## Step 7: Load and clean CBOE SKEW
#
# The SKEW index measures the perceived tail risk of the S&P 500 distribution.
# - SKEW = 100 → log-normal distribution (no tail risk)
# - SKEW > 100 → left tail is fatter than log-normal (crash risk premium)
# - Typical range: ~110-160
# - Spikes above 150 indicate extreme demand for OTM put protection

# %%
SKEW_PATH = OUTPUT_DIR / 'SKEW_History.csv'

print("=" * 80)
print("LOADING CBOE SKEW INDEX")
print("=" * 80)

skew_raw = pd.read_csv(SKEW_PATH)
print(f"Raw shape: {skew_raw.shape}")
print(f"Columns: {skew_raw.columns.tolist()}")
print(f"First 5 rows:")
print(skew_raw.head().to_string(index=False))

# ── Clean ────────────────────────────────────────────────────────────────────
skew = pd.DataFrame()
skew['date'] = pd.to_datetime(skew_raw['DATE'], errors='coerce')
skew['skew'] = pd.to_numeric(skew_raw['SKEW'], errors='coerce')

# Drop rows with invalid dates or values
skew = skew.dropna(subset=['date', 'skew']).reset_index(drop=True)

# Filter to sample period
skew = skew[
    (skew['date'] >= START) & (skew['date'] <= END)
].sort_values('date').reset_index(drop=True)

print(f"\nCleaned SKEW: {skew.shape}")
print(f"Date range: {skew['date'].min().date()} → {skew['date'].max().date()}")

# %% [markdown]
# ## Step 8: Compute derived SKEW factors

# %%
# ── SKEW deviation from "normal" ─────────────────────────────────────────────
skew['skew_excess'] = skew['skew'] - 100

# ── SKEW percentile (rolling 1-year) ────────────────────────────────────────
skew['skew_pctile_252d'] = (
    skew['skew']
    .rolling(252, min_periods=126)
    .apply(lambda x: (x.iloc[-1] > x.iloc[:-1]).mean() * 100, raw=False)
)

# ── SKEW momentum (5-day change) ────────────────────────────────────────────
skew['skew_chg_5d'] = skew['skew'].diff(5)

# ── SKEW 20-day moving average ──────────────────────────────────────────────
skew['skew_ma20'] = skew['skew'].rolling(20, min_periods=10).mean()

# ── SKEW relative to its moving average ─────────────────────────────────────
skew['skew_vs_ma20'] = skew['skew'] - skew['skew_ma20']

print(f"SKEW with derived factors: {skew.shape}")
print(f"Columns: {skew.columns.tolist()}")

# %% [markdown]
# ## Step 9: SKEW sanity checks

# %%
print("=" * 80)
print("CBOE SKEW SANITY CHECKS")
print("=" * 80)

print(f"\nskew (should be ~110-150 range):")
print(f"  Mean:   {skew['skew'].mean():.1f}")
print(f"  Median: {skew['skew'].median():.1f}")
print(f"  Min:    {skew['skew'].min():.1f}")
print(f"  Max:    {skew['skew'].max():.1f}")
print(f"  Std:    {skew['skew'].std():.1f}")

print(f"\nskew_excess (deviation from 100):")
print(f"  Mean:   {skew['skew_excess'].mean():.1f}")
print(f"  % positive (tail risk premium exists): "
      f"{(skew['skew_excess'] > 0).mean()*100:.1f}%")

print(f"\nNull counts:")
for c in skew.columns:
    if c == 'date':
        continue
    n = skew[c].isna().sum()
    if n > 0:
        print(f"  {c:<25s} {n:>5d} ({n/len(skew)*100:.1f}%)")

# %% [markdown]
# ## Step 10: Save SKEW

# %%
skew.to_parquet(
    OUTPUT_DIR / 'cboe_skew_daily.parquet', index=False, engine='pyarrow'
)
print(f"Saved {OUTPUT_DIR / 'cboe_skew_daily.parquet'}: {skew.shape}")


# %% [markdown]
# ---
# # PART 3: COMBINED OUTPUT
# ---

# %% [markdown]
# ## Step 11: Merge VIX futures + SKEW into single file

# %%
combined = vix_futures.merge(skew, on='date', how='outer')
combined = combined.sort_values('date').reset_index(drop=True)

# Filter to common sample period
combined = combined[
    (combined['date'] >= START) & (combined['date'] <= END)
].reset_index(drop=True)

print(f"Combined panel: {combined.shape}")
print(f"Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")
print(f"Columns: {combined.columns.tolist()}")

print(f"\nNull summary:")
for c in combined.columns:
    if c == 'date':
        continue
    n = combined[c].isna().sum()
    pct = n / len(combined) * 100
    print(f"  {c:<30s} {n:>5d} ({pct:.1f}%)")

combined.to_parquet(
    OUTPUT_DIR / 'vix_skew_combined.parquet', index=False, engine='pyarrow'
)
print(f"\nSaved {OUTPUT_DIR / 'vix_skew_combined.parquet'}: {combined.shape}")


# %% [markdown]
# ---
# # VERIFICATION & SUMMARY
# ---

# %%
print("=" * 80)
print("NOTEBOOK 11 — COMPLETE FACTOR INVENTORY")
print("=" * 80)

factor_cols = [c for c in combined.columns if c != 'date']
print(f"\nTotal factors: {len(factor_cols)}")
print(f"Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")
print(f"Trading days: {len(combined)}")

print(f"\nFactor list:")
for i, c in enumerate(factor_cols, 1):
    n_valid = combined[c].notna().sum()
    pct = n_valid / len(combined) * 100
    print(f"  {i:>2d}. {c:<30s} — {n_valid:>5d} valid obs ({pct:.1f}%)")

print(f"""
FACTOR DESCRIPTIONS:

  VIX Futures (from WRDS tr_ds_fut.dsfutcalcserval):
    vix_fut_front          — Front-month VIX futures settlement price
    vix_fut_second         — Second-month VIX futures settlement price
    vix_fut_volume         — Front-month daily volume
    vix_fut_oi             — Front-month open interest
    vix_term_spread        — VF2 - VF1 (contango > 0, backwardation < 0)
    vix_term_ratio         — VF2 / VF1 (normalised term structure)
    vix_fut_ret_1d         — Daily return on front-month VIX futures
    vix_term_spread_5d_chg — 5-day change in term structure spread

  CBOE SKEW Index (from cboe.com CSV):
    skew                   — CBOE SKEW Index level
    skew_excess            — SKEW minus 100 (tail risk premium)
    skew_pctile_252d       — Rolling 1-year percentile of SKEW
    skew_chg_5d            — 5-day change in SKEW
    skew_ma20              — 20-day moving average of SKEW
    skew_vs_ma20           — SKEW minus its 20-day MA

PIPELINE NOTES:
  • All data is daily, market-level. Merge onto trading calendar by date.
  • VIX futures basis (front_month - VIX_spot) should be computed in the
    merge pipeline where macro_daily.parquet (containing VIX close) is available.
  • No lookahead concerns: all values are known at market close of date t.
  • SKEW derived factors (pctile, ma20) use backward-looking windows only.
  • VIX futures launched March 26, 2004. Data before that date will be NaN.
""")


# %% [markdown]
# ## Cleanup

# %%
conn.close()
print("WRDS connection closed.")
print(f"\nFiles saved to {OUTPUT_DIR}:")
print(f"  vix_futures_daily.parquet")
print(f"  cboe_skew_daily.parquet")
print(f"  vix_skew_combined.parquet")

Loading library list...
Done
SCHEMA VERIFICATION: tr_ds_fut.dsfutcalcserval
Approximately 80494325 rows in tr_ds_fut.dsfutcalcserval.

tr_ds_fut.dsfutcalcserval — 8 columns:
  calcseriescode       NUMERIC(11, 0)      
  date_                DATE                
  open_                DOUBLE PRECISION    
  high                 DOUBLE PRECISION    
  low                  DOUBLE PRECISION    
  volume               DOUBLE PRECISION    
  settlement           DOUBLE PRECISION    
  openinterest         DOUBLE PRECISION    

Sample rows (calcseriescode=13396, front-month VIX, most recent):
 calcseriescode      date_  open_   high    low    volume  settlement  openinterest
        13396.0 2026-05-05   21.0  21.15  20.55  141918.0     21.0615      382113.0
        13396.0 2026-05-04  20.75   21.4   20.3  210862.0      21.045      380557.0
        13396.0 2026-05-01  20.55  20.92   20.4  140728.0     20.7897      374043.0
        13396.0 2026-04-30   20.5  20.67  19.37  144695.0     19.4774  

In [5]:
import wrds
import pandas as pd

conn = wrds.Connection(wrds_username='henrylavender')

# ── Query 1: Date range and row count for ALL VIX series codes ───────────────
# These are all the calcseriescode values from your metadata search
vix_codes = [
    2053, 6248, 10523, 11318, 12981, 13396,
    17679, 17680, 17681, 17682, 17683,
    18841, 18842, 18843,
    22013, 22014, 22015, 22016,
    22124, 22125, 22126, 22127, 22128,
    22773, 22774, 22775, 22776,
    22904, 22905, 22906, 22907,
]
codes_str = ','.join(str(c) for c in vix_codes)

print("=" * 100)
print("DIAGNOSTIC 1: Date coverage for all VIX calcseriescode values in dsfutcalcserval")
print("=" * 100)

coverage = conn.raw_sql(f"""
    SELECT v.calcseriescode,
           i.calcseriesname,
           MIN(v.date_) AS min_date,
           MAX(v.date_) AS max_date,
           COUNT(*) AS n_rows
    FROM tr_ds_fut.dsfutcalcserval v
    LEFT JOIN tr_ds_fut.wrds_cseries_info i
        ON v.calcseriescode = i.calcseriescode
    WHERE v.calcseriescode IN ({codes_str})
    GROUP BY v.calcseriescode, i.calcseriesname
    ORDER BY min_date ASC, n_rows DESC
""")

print(coverage.to_string(index=False))

# ── Query 2: Sample data from the series with the LONGEST history ────────────
print("\n" + "=" * 100)
print("DIAGNOSTIC 2: Sample rows from the series with longest history")
print("=" * 100)

if len(coverage) > 0:
    # Find the code with the earliest min_date
    best_code = coverage.loc[coverage['min_date'].idxmin(), 'calcseriescode']
    print(f"\nSeries with earliest data: calcseriescode = {best_code}")

    sample = conn.raw_sql(f"""
        SELECT *
        FROM tr_ds_fut.dsfutcalcserval
        WHERE calcseriescode = {best_code}
        ORDER BY date_ ASC
        LIMIT 10
    """)
    print("First 10 rows:")
    print(sample.to_string(index=False))

    sample_recent = conn.raw_sql(f"""
        SELECT *
        FROM tr_ds_fut.dsfutcalcserval
        WHERE calcseriescode = {best_code}
        ORDER BY date_ DESC
        LIMIT 5
    """)
    print("\nLast 5 rows:")
    print(sample_recent.to_string(index=False))

# ── Query 3: Check if the data lives in dsfutcalcserextval instead ───────────
print("\n" + "=" * 100)
print("DIAGNOSTIC 3: Check dsfutcalcserextval for VIX data")
print("=" * 100)

ext_coverage = conn.raw_sql(f"""
    SELECT calcseriescode,
           MIN(date_) AS min_date,
           MAX(date_) AS max_date,
           COUNT(*) AS n_rows,
           COUNT(DISTINCT item) AS n_items
    FROM tr_ds_fut.dsfutcalcserextval
    WHERE calcseriescode IN ({codes_str})
    GROUP BY calcseriescode
    ORDER BY n_rows DESC
""")
print(ext_coverage.to_string(index=False))

# ── Query 4: What items exist in the ext table? ─────────────────────────────
if len(ext_coverage) > 0:
    best_ext = ext_coverage.loc[ext_coverage['n_rows'].idxmax(), 'calcseriescode']
    print(f"\nItems for calcseriescode {best_ext} in dsfutcalcserextval:")
    items = conn.raw_sql(f"""
        SELECT DISTINCT item
        FROM tr_ds_fut.dsfutcalcserextval
        WHERE calcseriescode = {best_ext}
        ORDER BY item
    """)
    print(items.to_string(index=False))

conn.close()
print("\nDone. Paste this full output back and I'll write the corrected pull.")

Loading library list...
Done
DIAGNOSTIC 1: Date coverage for all VIX calcseriescode values in dsfutcalcserval
 calcseriescode                calcseriesname   min_date   max_date  n_rows
        17680.0            CFE-VIX INDEX TRc2 2004-03-26 2026-05-05    5564
         2053.0     CFE-VIX INDEX CONT. INDEX 2004-03-26 2026-05-05    5564
        17679.0            CFE-VIX INDEX TRc1 2004-03-26 2026-05-05    5563
        17681.0            CFE-VIX INDEX TRc3 2004-03-26 2026-05-05    5556
        22773.0   CFE-VIX INDEX CONT. 2ND VOL 2004-03-26 2026-05-05    5549
        22014.0   CFE-VIX INDEX CONT. 3RD FUT 2004-03-26 2026-05-05    5497
        22774.0   CFE-VIX INDEX CONT. 3RD VOL 2004-03-26 2026-05-05    5446
        22015.0   CFE-VIX INDEX CONT. 4TH FUT 2004-03-26 2026-05-05    5420
        22016.0   CFE-VIX INDEX CONT. 5TH FUT 2004-03-26 2026-05-05    5417
        22775.0   CFE-VIX INDEX CONT. 4TH VOL 2004-03-26 2026-05-05    5406
        22128.0      CFE-VIX INDEX CONT. TRAD 2004-03-

In [7]:
# %% [markdown]
# # Stage 11: VIX Futures Term Structure (WRDS) & CBOE SKEW Index
#
# Two orthogonal signals not captured anywhere else in the pipeline:
#
# 1. **VIX Futures Term Structure** — the spread between 2nd-month and front-month
#    VIX futures. Positive = contango (normal), negative = backwardation (panic).
#    This is distinct from VIX spot (which you already have) because it captures
#    how the market prices FUTURE volatility relative to CURRENT volatility.
#
# 2. **CBOE SKEW Index** — measures the perceived tail risk of the S&P 500 by
#    pricing the difference between OTM puts and OTM calls on SPX. High SKEW
#    = market is paying a premium for crash protection.
#
# Source tables:
#   WRDS: tr_ds_fut.dsfutcalcserval
#     - calcseriescode 17679 = CFE-VIX INDEX TRc1 (front-month continuous)
#     - calcseriescode 17680 = CFE-VIX INDEX TRc2 (second-month continuous)
#     These are the Thomson Reuters continuous contract series, stitched from
#     individual expiring contracts. Full history from 2004-03-26 (VIX futures
#     launch date) to present.
#
#     NOTE: calcseriescode 13396/22013 (labelled "CONTINUOUS" / "CONT. 2ND FUT")
#     only have data from May 2024 onwards — they are a recently created series.
#     The TRc1/TRc2 series (17679/17680) contain the full 20+ year history.
#
#   Local CSV: CBOE SKEW_History.csv (downloaded from cboe.com)
#
# Output:
#   Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/vix_futures_daily.parquet
#   Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/cboe_skew_daily.parquet
#   Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/vix_skew_combined.parquet

# %% [markdown]
# ## Setup

# %%
import wrds
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

conn = wrds.Connection(wrds_username='henrylavender')

START = '2004-01-01'
END   = '2024-12-31'

# The correct calcseriescode values with full history (2004-03-26 onwards):
FRONT_MONTH_CODE = 17679   # CFE-VIX INDEX TRc1 (front-month continuous)
SECOND_MONTH_CODE = 17680  # CFE-VIX INDEX TRc2 (second-month continuous)

# %% [markdown]
# ---
# # PART 1: VIX FUTURES TERM STRUCTURE
# ---

# %% [markdown]
# ## Step 1: Verify date coverage before full pull

# %%
print("=" * 80)
print("DATE COVERAGE CHECK")
print("=" * 80)

for code, label in [(FRONT_MONTH_CODE, 'Front-month TRc1'),
                     (SECOND_MONTH_CODE, 'Second-month TRc2')]:
    info = conn.raw_sql(f"""
        SELECT MIN(date_) AS min_date, MAX(date_) AS max_date, COUNT(*) AS n_rows
        FROM tr_ds_fut.dsfutcalcserval
        WHERE calcseriescode = {code}
    """)
    print(f"\n  {label} (code {code}):")
    print(f"    Date range: {info['min_date'].iloc[0]} → {info['max_date'].iloc[0]}")
    print(f"    Total rows: {info['n_rows'].iloc[0]:,}")

# %% [markdown]
# ## Step 2: Pull VIX futures data
#
# Pull from `dsfutcalcserval` — clean 8-column table, one row per
# (calcseriescode, date_). Date column is `date_`, price column is `settlement`.

# %%
print("\n" + "=" * 80)
print("PULLING VIX FUTURES FROM tr_ds_fut.dsfutcalcserval")
print("=" * 80)

# ── Front-month ──────────────────────────────────────────────────────────────
print(f"\nPulling front-month TRc1 (calcseriescode={FRONT_MONTH_CODE})...")
vix_front_raw = conn.raw_sql(f"""
    SELECT date_, open_, high, low, volume, settlement, openinterest
    FROM tr_ds_fut.dsfutcalcserval
    WHERE calcseriescode = {FRONT_MONTH_CODE}
      AND date_ BETWEEN '{START}' AND '{END}'
    ORDER BY date_
""")
print(f"  Rows: {len(vix_front_raw):,}")
print(f"  Date range: {vix_front_raw['date_'].min()} → {vix_front_raw['date_'].max()}")
print(f"  Settlement range: {vix_front_raw['settlement'].min():.2f} – "
      f"{vix_front_raw['settlement'].max():.2f}")

# ── Second-month ─────────────────────────────────────────────────────────────
print(f"\nPulling second-month TRc2 (calcseriescode={SECOND_MONTH_CODE})...")
vix_second_raw = conn.raw_sql(f"""
    SELECT date_, open_, high, low, volume, settlement, openinterest
    FROM tr_ds_fut.dsfutcalcserval
    WHERE calcseriescode = {SECOND_MONTH_CODE}
      AND date_ BETWEEN '{START}' AND '{END}'
    ORDER BY date_
""")
print(f"  Rows: {len(vix_second_raw):,}")
print(f"  Date range: {vix_second_raw['date_'].min()} → {vix_second_raw['date_'].max()}")
print(f"  Settlement range: {vix_second_raw['settlement'].min():.2f} – "
      f"{vix_second_raw['settlement'].max():.2f}")

# ── Validate ─────────────────────────────────────────────────────────────────
assert len(vix_front_raw) > 4000, (
    f"Expected ~5,000+ front-month rows, got {len(vix_front_raw)}. "
    f"Check calcseriescode {FRONT_MONTH_CODE}."
)
assert len(vix_second_raw) > 4000, (
    f"Expected ~5,000+ second-month rows, got {len(vix_second_raw)}. "
    f"Check calcseriescode {SECOND_MONTH_CODE}."
)
print(f"\n✓ Both series have sufficient coverage.")

# %% [markdown]
# ## Step 3: Build the VIX futures panel

# %%
print("\n" + "=" * 80)
print("BUILDING VIX FUTURES PANEL")
print("=" * 80)

# ── Clean front-month ────────────────────────────────────────────────────────
vf1 = pd.DataFrame()
vf1['date'] = pd.to_datetime(vix_front_raw['date_'])
vf1['vix_fut_front'] = pd.to_numeric(vix_front_raw['settlement'], errors='coerce')
vf1['vix_fut_volume'] = pd.to_numeric(vix_front_raw['volume'], errors='coerce')
vf1['vix_fut_oi'] = pd.to_numeric(vix_front_raw['openinterest'], errors='coerce')

# Drop rows where settlement is NaN or zero
vf1 = vf1[vf1['vix_fut_front'].notna() & (vf1['vix_fut_front'] > 0)]
vf1 = vf1.drop_duplicates(subset='date', keep='first')
print(f"Front-month cleaned: {len(vf1):,} rows")
print(f"  Date range: {vf1['date'].min().date()} → {vf1['date'].max().date()}")

# ── Clean second-month ───────────────────────────────────────────────────────
vf2 = pd.DataFrame()
vf2['date'] = pd.to_datetime(vix_second_raw['date_'])
vf2['vix_fut_second'] = pd.to_numeric(vix_second_raw['settlement'], errors='coerce')

vf2 = vf2[vf2['vix_fut_second'].notna() & (vf2['vix_fut_second'] > 0)]
vf2 = vf2.drop_duplicates(subset='date', keep='first')
print(f"Second-month cleaned: {len(vf2):,} rows")
print(f"  Date range: {vf2['date'].min().date()} → {vf2['date'].max().date()}")

# ── Merge on date ────────────────────────────────────────────────────────────
vix_futures = vf1.merge(vf2, on='date', how='outer').sort_values('date').reset_index(drop=True)
print(f"\nMerged panel: {len(vix_futures):,} rows")
print(f"  Date range: {vix_futures['date'].min().date()} → {vix_futures['date'].max().date()}")

# %% [markdown]
# ## Step 4: Compute derived factors

# %%
# ── VIX futures term structure spread ────────────────────────────────────────
# Positive = contango (normal market, fear is priced as transient)
# Negative = backwardation (panic, near-term fear exceeds longer-term)
vix_futures['vix_term_spread'] = (
    vix_futures['vix_fut_second'] - vix_futures['vix_fut_front']
)

# ── VIX futures term structure ratio ─────────────────────────────────────────
# Normalised measure: VF2/VF1. >1 = contango, <1 = backwardation.
vix_futures['vix_term_ratio'] = (
    vix_futures['vix_fut_second'] /
    vix_futures['vix_fut_front'].replace(0, np.nan)
)

# ── Daily return on front-month VIX futures ──────────────────────────────────
vix_futures['vix_fut_ret_1d'] = vix_futures['vix_fut_front'].pct_change()

# ── 5-day rolling change in term structure ───────────────────────────────────
vix_futures['vix_term_spread_5d_chg'] = vix_futures['vix_term_spread'].diff(5)

# ── Filter to sample period ──────────────────────────────────────────────────
vix_futures = vix_futures[
    (vix_futures['date'] >= START) & (vix_futures['date'] <= END)
].reset_index(drop=True)

print(f"VIX futures panel (final): {vix_futures.shape}")
print(f"Date range: {vix_futures['date'].min().date()} → {vix_futures['date'].max().date()}")
print(f"Columns: {vix_futures.columns.tolist()}")

# %% [markdown]
# ## Step 5: VIX futures sanity checks

# %%
print("=" * 80)
print("VIX FUTURES SANITY CHECKS")
print("=" * 80)

for col in ['vix_fut_front', 'vix_fut_second']:
    vals = vix_futures[col].dropna()
    print(f"\n{col} ({len(vals):,} valid obs):")
    print(f"  Mean:   {vals.mean():.2f}")
    print(f"  Median: {vals.median():.2f}")
    print(f"  Min:    {vals.min():.2f}")
    print(f"  Max:    {vals.max():.2f}")

ts = vix_futures['vix_term_spread'].dropna()
print(f"\nvix_term_spread ({len(ts):,} valid obs):")
print(f"  Mean:   {ts.mean():.3f}  (should be positive on average = contango)")
print(f"  Median: {ts.median():.3f}")
print(f"  Min:    {ts.min():.3f}  (should be deeply negative in crisis)")
print(f"  Max:    {ts.max():.3f}")
print(f"  % negative (backwardation): {(ts < 0).mean()*100:.1f}%")

tr = vix_futures['vix_term_ratio'].dropna()
print(f"\nvix_term_ratio ({len(tr):,} valid obs):")
print(f"  Mean:   {tr.mean():.4f}")
print(f"  Median: {tr.median():.4f}")
print(f"  Min:    {tr.min():.4f}")
print(f"  Max:    {tr.max():.4f}")

print(f"\nNull counts:")
for c in vix_futures.columns:
    if c == 'date':
        continue
    n = vix_futures[c].isna().sum()
    pct = n / len(vix_futures) * 100
    print(f"  {c:<30s} {n:>5d} ({pct:.1f}%)")

# ── Crisis spot-checks ──────────────────────────────────────────────────────
print(f"\n--- Crisis Spot-Checks ---")
for label, start_d, end_d in [
    ('GFC Sep-Oct 2008',        '2008-09-01', '2008-10-31'),
    ('Volmageddon Feb 2018',    '2018-02-01', '2018-02-15'),
    ('COVID Mar 2020',          '2020-03-01', '2020-03-31'),
    ('Yen carry unwind Aug 24', '2024-08-01', '2024-08-15'),
]:
    mask = (vix_futures['date'] >= start_d) & (vix_futures['date'] <= end_d)
    subset = vix_futures[mask]
    if len(subset) > 0 and subset['vix_term_spread'].notna().any():
        print(f"\n  {label}:")
        print(f"    vix_fut_front: {subset['vix_fut_front'].min():.1f} – "
              f"{subset['vix_fut_front'].max():.1f}")
        print(f"    vix_term_spread: {subset['vix_term_spread'].min():.2f} – "
              f"{subset['vix_term_spread'].max():.2f}")
        print(f"    (Expect backwardation = negative spread)")
    else:
        print(f"\n  {label}: no data in this range")

# %% [markdown]
# ## Step 6: Save VIX futures

# %%
vix_futures.to_parquet(
    OUTPUT_DIR / 'vix_futures_daily.parquet', index=False, engine='pyarrow'
)
print(f"Saved {OUTPUT_DIR / 'vix_futures_daily.parquet'}: {vix_futures.shape}")


# %% [markdown]
# ---
# # PART 2: CBOE SKEW INDEX
# ---

# %% [markdown]
# ## Step 7: Load and clean CBOE SKEW
#
# The SKEW index measures the perceived tail risk of the S&P 500 distribution.
# - SKEW = 100 → log-normal distribution (no tail risk)
# - SKEW > 100 → left tail is fatter than log-normal (crash risk premium)
# - Typical range: ~110-160
# - Spikes above 150 indicate extreme demand for OTM put protection

# %%
SKEW_PATH = OUTPUT_DIR / 'SKEW_History.csv'

print("=" * 80)
print("LOADING CBOE SKEW INDEX")
print("=" * 80)

skew_raw = pd.read_csv(SKEW_PATH)
print(f"Raw shape: {skew_raw.shape}")
print(f"Columns: {skew_raw.columns.tolist()}")
print(f"First 5 rows:")
print(skew_raw.head().to_string(index=False))

# ── Clean ────────────────────────────────────────────────────────────────────
skew = pd.DataFrame()
skew['date'] = pd.to_datetime(skew_raw['DATE'], errors='coerce')
skew['skew'] = pd.to_numeric(skew_raw['SKEW'], errors='coerce')

skew = skew.dropna(subset=['date', 'skew']).reset_index(drop=True)

skew = skew[
    (skew['date'] >= START) & (skew['date'] <= END)
].sort_values('date').reset_index(drop=True)

print(f"\nCleaned SKEW: {skew.shape}")
print(f"Date range: {skew['date'].min().date()} → {skew['date'].max().date()}")

# %% [markdown]
# ## Step 8: Compute derived SKEW factors

# %%
# ── SKEW deviation from "normal" ─────────────────────────────────────────────
skew['skew_excess'] = skew['skew'] - 100

# ── SKEW percentile (rolling 1-year) ────────────────────────────────────────
skew['skew_pctile_252d'] = (
    skew['skew']
    .rolling(252, min_periods=126)
    .apply(lambda x: (x.iloc[-1] > x.iloc[:-1]).mean() * 100, raw=False)
)

# ── SKEW momentum (5-day change) ────────────────────────────────────────────
skew['skew_chg_5d'] = skew['skew'].diff(5)

# ── SKEW 20-day moving average ──────────────────────────────────────────────
skew['skew_ma20'] = skew['skew'].rolling(20, min_periods=10).mean()

# ── SKEW relative to its moving average ─────────────────────────────────────
skew['skew_vs_ma20'] = skew['skew'] - skew['skew_ma20']

print(f"SKEW with derived factors: {skew.shape}")
print(f"Columns: {skew.columns.tolist()}")

# %% [markdown]
# ## Step 9: SKEW sanity checks

# %%
print("=" * 80)
print("CBOE SKEW SANITY CHECKS")
print("=" * 80)

print(f"\nskew (should be ~110-150 range):")
print(f"  Mean:   {skew['skew'].mean():.1f}")
print(f"  Median: {skew['skew'].median():.1f}")
print(f"  Min:    {skew['skew'].min():.1f}")
print(f"  Max:    {skew['skew'].max():.1f}")
print(f"  Std:    {skew['skew'].std():.1f}")

print(f"\nskew_excess (deviation from 100):")
print(f"  Mean:   {skew['skew_excess'].mean():.1f}")
print(f"  % positive (tail risk premium exists): "
      f"{(skew['skew_excess'] > 0).mean()*100:.1f}%")

print(f"\nNull counts:")
for c in skew.columns:
    if c == 'date':
        continue
    n = skew[c].isna().sum()
    if n > 0:
        print(f"  {c:<25s} {n:>5d} ({n/len(skew)*100:.1f}%)")

# %% [markdown]
# ## Step 10: Save SKEW

# %%
skew.to_parquet(
    OUTPUT_DIR / 'cboe_skew_daily.parquet', index=False, engine='pyarrow'
)
print(f"Saved {OUTPUT_DIR / 'cboe_skew_daily.parquet'}: {skew.shape}")


# %% [markdown]
# ---
# # PART 3: COMBINED OUTPUT
# ---

# %% [markdown]
# ## Step 11: Merge VIX futures + SKEW into single file

# %%
combined = vix_futures.merge(skew, on='date', how='outer')
combined = combined.sort_values('date').reset_index(drop=True)

combined = combined[
    (combined['date'] >= START) & (combined['date'] <= END)
].reset_index(drop=True)

print(f"Combined panel: {combined.shape}")
print(f"Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")
print(f"Columns: {combined.columns.tolist()}")

print(f"\nNull summary:")
for c in combined.columns:
    if c == 'date':
        continue
    n = combined[c].isna().sum()
    pct = n / len(combined) * 100
    print(f"  {c:<30s} {n:>5d} ({pct:.1f}%)")

combined.to_parquet(
    OUTPUT_DIR / 'vix_skew_combined.parquet', index=False, engine='pyarrow'
)
print(f"\nSaved {OUTPUT_DIR / 'vix_skew_combined.parquet'}: {combined.shape}")


# %% [markdown]
# ---
# # VERIFICATION & SUMMARY
# ---

# %%
print("=" * 80)
print("NOTEBOOK 11 — COMPLETE FACTOR INVENTORY")
print("=" * 80)

factor_cols = [c for c in combined.columns if c != 'date']
print(f"\nTotal factors: {len(factor_cols)}")
print(f"Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")
print(f"Trading days: {len(combined)}")

print(f"\nFactor list:")
for i, c in enumerate(factor_cols, 1):
    n_valid = combined[c].notna().sum()
    pct = n_valid / len(combined) * 100
    print(f"  {i:>2d}. {c:<30s} — {n_valid:>5d} valid obs ({pct:.1f}%)")

print(f"""
FACTOR DESCRIPTIONS:

  VIX Futures (from WRDS tr_ds_fut.dsfutcalcserval):
    Source: calcseriescode {FRONT_MONTH_CODE} (TRc1) and {SECOND_MONTH_CODE} (TRc2)
    These are Thomson Reuters continuous contract series, stitched from
    individual expiring VIX futures contracts.

    vix_fut_front          — Front-month VIX futures settlement price
    vix_fut_second         — Second-month VIX futures settlement price
    vix_fut_volume         — Front-month daily volume (contracts)
    vix_fut_oi             — Front-month daily open interest (contracts)
    vix_term_spread        — VF2 - VF1 (contango > 0, backwardation < 0)
    vix_term_ratio         — VF2 / VF1 (normalised term structure)
    vix_fut_ret_1d         — Daily return on front-month VIX futures
    vix_term_spread_5d_chg — 5-day change in term structure spread

  CBOE SKEW Index (from cboe.com CSV):
    skew                   — CBOE SKEW Index level
    skew_excess            — SKEW minus 100 (tail risk premium)
    skew_pctile_252d       — Rolling 1-year percentile of SKEW
    skew_chg_5d            — 5-day change in SKEW
    skew_ma20              — 20-day moving average of SKEW
    skew_vs_ma20           — SKEW minus its 20-day MA

PIPELINE NOTES:
  • All data is daily, market-level. Merge onto trading calendar by date.
  • VIX futures basis (front_month - VIX_spot) should be computed in the
    merge pipeline where macro_daily.parquet (containing VIX close) is available.
  • No lookahead concerns: all values are known at market close of date t.
  • SKEW derived factors (pctile, ma20) use backward-looking windows only.
  • VIX futures launched March 26, 2004. Data before that date will be NaN.
""")


# %% [markdown]
# ## Cleanup

# %%
conn.close()
print("WRDS connection closed.")
print(f"\nFiles saved to {OUTPUT_DIR}:")
print(f"  vix_futures_daily.parquet")
print(f"  cboe_skew_daily.parquet")
print(f"  vix_skew_combined.parquet")

Loading library list...
Done
DATE COVERAGE CHECK

  Front-month TRc1 (code 17679):
    Date range: 2004-03-26 → 2026-05-05
    Total rows: 5,563

  Second-month TRc2 (code 17680):
    Date range: 2004-03-26 → 2026-05-05
    Total rows: 5,564

PULLING VIX FUTURES FROM tr_ds_fut.dsfutcalcserval

Pulling front-month TRc1 (calcseriescode=17679)...
  Rows: 5,227
  Date range: 2004-03-26 → 2024-12-31
  Settlement range: 9.88 – 72.62

Pulling second-month TRc2 (calcseriescode=17680)...
  Rows: 5,228
  Date range: 2004-03-26 → 2024-12-31
  Settlement range: 11.32 – 70.47

✓ Both series have sufficient coverage.

BUILDING VIX FUTURES PANEL
Front-month cleaned: 5,226 rows
  Date range: 2004-03-26 → 2024-12-31
Second-month cleaned: 5,227 rows
  Date range: 2004-03-26 → 2024-12-31

Merged panel: 5,227 rows
  Date range: 2004-03-26 → 2024-12-31


C:\Users\Henry\AppData\Local\Temp\ipykernel_40808\1751940468.py:184: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  vix_futures['vix_fut_ret_1d'] = vix_futures['vix_fut_front'].pct_change()


VIX futures panel (final): (5227, 9)
Date range: 2004-03-26 → 2024-12-31
Columns: ['date', 'vix_fut_front', 'vix_fut_volume', 'vix_fut_oi', 'vix_fut_second', 'vix_term_spread', 'vix_term_ratio', 'vix_fut_ret_1d', 'vix_term_spread_5d_chg']
VIX FUTURES SANITY CHECKS

vix_fut_front (5,226 valid obs):
  Mean:   19.53
  Median: 17.23
  Min:    9.88
  Max:    72.62

vix_fut_second (5,227 valid obs):
  Mean:   20.39
  Median: 18.32
  Min:    11.32
  Max:    70.47

vix_term_spread (5,226 valid obs):
  Mean:   0.868  (should be positive on average = contango)
  Median: 1.049
  Min:    -21.100  (should be deeply negative in crisis)
  Max:    6.525
  % negative (backwardation): 15.7%

vix_term_ratio (5,226 valid obs):
  Mean:   1.0591
  Median: 1.0654
  Min:    0.6697
  Max:    1.3375

Null counts:
  vix_fut_front                      1 (0.0%)
  vix_fut_volume                    59 (1.1%)
  vix_fut_oi                        12 (0.2%)
  vix_fut_second                     0 (0.0%)
  vix_term_spread